# PHASE 7: Proof Tool Integration

**Implements SEC-006: Replace proof tool stubs with real execution**

## Objective

Integrate actual proof verifiers (Agda, Ada/SPARK, Lean 4, Z3) instead of stub predicates.

| Tool | Language | Purpose | Status |
|------|----------|---------|--------|
| **Agda** | Proof Assistant | Type checking, invariant proofs | PHASE 7.1 |
| **Ada/SPARK** | Formal Methods | Loop & invariant verification | PHASE 7.2 |
| **Lean 4** | Proof Language | Curry-Howard isomorphism | PHASE 7.3 |
| **Z3** | SMT Solver | Symbolic execution | PHASE 7.4 |


## PHASE 7.1: Agda Integration

### Task: Invoke Agda for type checking

**File:** `crates/proof-validator/src/adapters/agda_adapter.rs`

```rust
pub struct AgdaAdapter {
    agda_bin: PathBuf,
    library_path: Vec<PathBuf>,
}

impl AgdaAdapter {
    pub async fn verify_type(&self, source: &str, expected_type: &str) -> Result<ProofStatus> {
        // 1. Write source to temp file
        // 2. Invoke: agda --check source.agda
        // 3. Parse output (OK / ERROR)
        // 4. Return ProofStatus::Proved or ProofStatus::Disproved
    }
}
```

**Evidence:**
- Creates temp Agda files
- Spawns agda process
- Parses exit code + stderr
- Returns deterministic result

## PHASE 7.2: Ada/SPARK Integration

### Task: SPARK verifier for loop invariants

**File:** `crates/proof-validator/src/adapters/spark_adapter.rs`

```rust
pub struct SparkAdapter {
    spark_bin: PathBuf,
    timeout_secs: u64,
}

impl SparkAdapter {
    pub async fn verify_loop_invariant(&self, ada_source: &str, invariant: &str) -> Result<ProofStatus> {
        // 1. Extract loop from source
        // 2. Annotate with --# loop_invariant pragma
        // 3. Run: gnatprove -P project.gpr
        // 4. Check proof results
        // 5. Return Proved / Disproved / Timeout
    }
}
```

**Evidence:**
- Parses Ada loop syntax
- Instruments with SPARK pragmas
- Invokes gnatprove
- Deterministic timeout handling

## PHASE 7.3: Lean 4 Integration

### Task: Curry-Howard type checking

**File:** `crates/proof-validator/src/adapters/lean4_adapter.rs`

```rust
pub struct Lean4Adapter {
    lean_bin: PathBuf,
    stdlib_path: PathBuf,
}

impl Lean4Adapter {
    pub async fn verify_curry_howard(&self, lean_proof: &str, theorem: &str) -> Result<ProofStatus> {
        // 1. Write proof to .lean file
        // 2. Run: lean --check proof.lean
        // 3. Parse Lean diagnostics
        // 4. Match theorem signature
        // 5. Return Proved or error message
    }
}
```

**Evidence:**
- Validates proof term structure
- Type-checks against theorem statement
- Extracts proof witness
- Returns serializable proof object

## PHASE 7.4: Z3 SMT Solver Integration

### Task: Symbolic execution for invariant extraction

**File:** `crates/proof-validator/src/adapters/z3_adapter.rs`

```rust
pub struct Z3Adapter {
    z3_ctx: z3::Context,
}

impl Z3Adapter {
    pub fn verify_invariant(&self, formula: &str, invariant: &str) -> Result<ProofStatus> {
        // 1. Parse formula to Z3 expression
        // 2. Assert (NOT invariant)
        // 3. Check satisfiability
        // 4. If UNSAT: invariant proven
        // 5. If SAT: return counterexample
    }
}
```

**Evidence:**
- Uses z3-rs bindings
- SMT-LIB syntax support
- Counterexample extraction
- Proof object construction

## Implementation Checklist

### PHASE 7.1: Agda
- [ ] Create `agda_adapter.rs`
- [ ] Implement `verify_type()` method
- [ ] Add temp file creation
- [ ] Parse agda exit codes
- [ ] Unit tests (3 test cases)
- [ ] Integration test with sample proof

### PHASE 7.2: Ada/SPARK
- [ ] Create `spark_adapter.rs`
- [ ] Implement `verify_loop_invariant()`
- [ ] Pragma injection
- [ ] gnatprove subprocess handling
- [ ] Timeout + cleanup
- [ ] Unit tests (2 test cases)

### PHASE 7.3: Lean 4
- [ ] Create `lean4_adapter.rs`
- [ ] Implement `verify_curry_howard()`
- [ ] .lean file writing
- [ ] Lean diagnostics parsing
- [ ] Proof term extraction
- [ ] Unit tests (3 test cases)

### PHASE 7.4: Z3
- [ ] Create `z3_adapter.rs`
- [ ] Implement `verify_invariant()`
- [ ] SMT-LIB formula construction
- [ ] Satisfiability checking
- [ ] Counterexample extraction
- [ ] Unit tests (4 test cases)

## Proof Obligations

### InvariantPreservation
```
∀ state: State.
  invariant(state) ∧ transition(state, state') →
  invariant(state')
```

**Verified by:** Z3 SMT solver (symbolic execution)

### SemanticPreservation
```
∀ source: String.
  semantics(compile(source)) = semantics(source)
```

**Verified by:** Lean 4 (Curry-Howard proof)

### LoopInvariantMaintenance
```
∀ i: Nat.
  loop_invariant(i) ∧ loop_condition(i) →
  loop_invariant(i+1)
```

**Verified by:** Ada/SPARK (gnatprove)

### ReceiptChainIntegrity
```
∀ r1, r2: Receipt.
  r1.sequence < r2.sequence ∧
  r2.previous_hash = r1.hash →
  chain_valid(r1, r2)
```

**Verified by:** Agda (type checking)

## Release Gate Integration

### Prolog Gate (logic/rules/release_ready.pl)

```prolog
% Before Phase 7: stubs always return true
proof_verified(_, stub, true) :- !.

% After Phase 7: real verifiers required
proof_verified(Obligation, agda, true) :-
    agda_adapter:verify_type(Obligation, _).

proof_verified(Obligation, spark, true) :-
    spark_adapter:verify_loop_invariant(Obligation, _).

proof_verified(Obligation, lean4, true) :-
    lean4_adapter:verify_curry_howard(Obligation, _).

proof_verified(Obligation, z3, true) :-
    z3_adapter:verify_invariant(Obligation, _).
```

### Release Readiness Check

```prolog
release_ready(proof_obligations_discharged) :-
    forall(
        proof_obligation(Obligation, Tool, _),
        proof_verified(Obligation, Tool, true)
    ).
```

## Execution Flow

### Cell Execution → Proof Validation → Receipt

```
1. Notebook cell executes (user code)
   ↓
2. Invariant extractor infers proof obligations
   ↓
3. Dispatch to appropriate verifier:
   - Agda for type safety
   - Ada/SPARK for loop invariants
   - Lean 4 for semantic preservation
   - Z3 for symbolic execution
   ↓
4. Verifier returns ProofStatus (Proved/Disproved/Manual/Error)
   ↓
5. If all proofs Proved:
   - Create WORM-sealed receipt
   - Link to receipt chain
   - Mark cell as VERIFIED
   ↓
6. If any proof fails:
   - Rollback cell
   - Return error to notebook
   - Operator must fix and retry
```

## Success Criteria

✅ **PHASE 7 Complete When:**

1. All 4 adapters implemented (Agda, SPARK, Lean 4, Z3)
2. Each adapter has 2-4 unit tests (12+ total)
3. Each adapter can be invoked from release gate
4. Prolog predicates updated to call real verifiers
5. End-to-end test: execute cell → verify proof → seal receipt
6. No stubs remaining (all ProofStatus calls reach actual tools)
7. Timeout + error handling for each tool
8. 37+ existing tests still passing
9. All commits pushed to GitHub
10. Notebook artifact committed (`.ipynb` file)

**Status: READY FOR IMPLEMENTATION**